In [1]:

# physical_graph_path = "scenarios/cavia/1_26_solution_v1/physical_graph.graphml"
# app_graph_path = "scenarios/cavia/1_26_solution_v1/ms/1MMM.graphml"
# pkl_path = "scenarios/cavia/1_26_solution_v1/var_coeff_values_1MMM_slss.pkl"

import os

from adapters.cavia.cavia_scenario_loader import CaviaScenarioLoader
from adapters.cavia.find_valid_scenarios import find_or_load_scenarios
from edge_sim_py.component_manager import ComponentManager

HOME_DIR = os.path.expanduser("~")
BASE_PATH = os.path.join(HOME_DIR, "Desktop")
PKL_PATH = os.path.join(BASE_PATH, "CAVIA", "src", "LR")

scenarios = find_or_load_scenarios(PKL_PATH)
scenario = scenarios[0]
print("Loading scenario:", scenario + "\n")

physical_graph_path = os.path.join(BASE_PATH, scenario, "physical_graph.graphml")
app_graph_path = os.path.join(BASE_PATH, scenario, "ms/1MMM.graphml")
pkl_path = os.path.join(BASE_PATH, scenario, "var_coeff_values_1MMM_slss.pkl")

topology, app, user = CaviaScenarioLoader(
    physical_graph_path=physical_graph_path,
    app_graph_path=app_graph_path,
    pkl_path=pkl_path,
).build_scenario()

print("Topology:", topology)
print("Application:", app)
print("User:", user)

Found valid_scenarios_cache.json in: /home/damiano/Desktop/EdgeSimPy/adapters/cavia/valid_scenarios_cache.json
Loading scenario: CAVIA/src/LR/1_26_solution_v1

Topology: Topology_1
Application: Application_1
User: User_1


In [2]:
from edge_sim_py.components.user_access_patterns.circular_duration_and_interval_access_pattern import CircularDurationAndIntervalAccessPattern
from edge_sim_py.simulator import Simulator

CircularDurationAndIntervalAccessPattern(user=user, app=app, start=1, duration_values=[1], interval_values=[100])
dataset = ComponentManager.export_scenario(save_to_file=True, file_name="1_26_solution_v1")

def my_algorithm(parameters):
    return

def static_dummy_mobility(user):
    user.coordinates_trace.append(user.coordinates)

# Instantiating the simulator
simulator = Simulator(
    dump_interval=1,
    tick_unit="seconds",
    tick_duration=1,
    stopping_criterion= lambda model: model.schedule.steps == 100,
    resource_management_algorithm=my_algorithm,
    user_defined_functions=[static_dummy_mobility]
)

# Loading our dataset
simulator.initialize(input_file="datasets/1_26_solution_v1.json")

# Running the simulation
simulator.run_model()

In [3]:
import msgpack
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', 1000)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 1000)

def highlight_rows(row):
    if row["Time Step"] % 2 == 0:
        return ["background-color: #222222; color: white"] * len(row)
    else:
        return ["background-color: #333333; color: white"] * len(row)

def load_simulation_logs(logs_path="logs"):
    datasets = {}
    directory = Path(logs_path)
    
    if not directory.exists():
        print(f"Errore: La cartella {logs_path} non esiste.")
        return datasets

    for file_path in directory.glob("*.msgpack"):
        with open(file_path, "rb") as f:
            data = msgpack.unpackb(f.read(), strict_map_key=False)
            datasets[file_path.stem] = pd.DataFrame(data)
            
    return datasets

datasets = load_simulation_logs()

In [4]:
print(datasets.keys())
#datasets["Application"].copy().style.apply(highlight_rows, axis=1)
#datasets["EdgeServer"].copy().style.apply(highlight_rows, axis=1)
#d = datasets["User"].copy().style.apply(highlight_rows, axis=1)
datasets["User"].head(2).style.apply(highlight_rows, axis=1)
#datasets["Service"].copy().style.apply(highlight_rows, axis=1)
#datasets["NetworkFlow"].copy().style.apply(highlight_rows, axis=1)

dict_keys(['Service', 'Application', 'EdgeServer', 'User', 'DataPacket', 'NetworkSwitch'])


,Object,Time Step,Instance ID,Coordinates,Base Station,Delays,Delay SLAs,Communication Paths,Making Requests,Access History
0,User_1,0,1,"[15, 1]","BaseStation_8 ([15, 1])",{'1': None},{'1': 67.278},{},{'1': {'1': True}},"{'1': [{'start': 1, 'end': 1, 'duration': 1, 'waiting_time': 0, 'access_time': 0, 'interval': 100, 'next_access': 102}]}"
1,User_1,1,1,"[15, 1]","BaseStation_8 ([15, 1])",{'1': 27},{'1': 67.278},"{'1': [[8], [8], [8, 21, 16], [16, 21, 24, 2], [2, 10, 4, 8]]}","{'1': {'1': True, '2': False}}","{'1': [{'start': 1, 'end': 1, 'duration': 1, 'waiting_time': 0, 'access_time': 1, 'interval': 100, 'next_access': 102}]}"


In [5]:
#datasets["DataPacket"].copy().style.apply(highlight_rows, axis=1)

df_finished = datasets["DataPacket"][datasets["DataPacket"]["Status"] == "finished"].copy()
#df_finished.head(1).style.apply(highlight_rows, axis=1)
df_finished.columns


Index(['Object', 'Time Step', 'Id', 'User', 'Application', 'Size', 'Status', 'Queue Delay', 'Transmission Delay', 'Processing Delay', 'Propagation Delay', 'Total Delay', 'Total Path', 'Hops'], dtype='object')

In [9]:
#df_finished[["Time Step", "Total Delay"]].head(1).style.apply(highlight_rows, axis=1)
df_finished.drop(columns=["Hops"]).head(1).style.apply(highlight_rows, axis=1)

,Object,Time Step,Id,User,Application,Size,Status,Queue Delay,Transmission Delay,Processing Delay,Propagation Delay,Total Delay,Total Path
11,DataPacket_1,12,1,1,1,0,finished,0,8,0,27,35,"[[8], [8], [8, 21, 16], [16, 21, 24, 2], [2, 10, 4, 8]]"


In [12]:
df_finished[["Time Step", "Hops"]].head(1).style.apply(highlight_rows, axis=1)

,Time Step,Hops
11,12,"[{'hop_index': 0, 'link_index': 0, 'source': 8, 'target': 8, 'start_time': 1, 'end_time': 1, 'queue_delay': 0, 'transmission_delay': 0, 'processing_delay': 0, 'propagation_delay': 0, 'min_bandwidth': 0, 'max_bandwidth': 0, 'avg_bandwidth': 0, 'data_input': 75, 'data_output': 75}, {'hop_index': 1, 'link_index': 0, 'source': 8, 'target': 8, 'start_time': 1, 'end_time': 1, 'queue_delay': 0, 'transmission_delay': 0, 'processing_delay': 0, 'propagation_delay': 0, 'min_bandwidth': 0, 'max_bandwidth': 0, 'avg_bandwidth': 0, 'data_input': 75, 'data_output': 50}, {'hop_index': 2, 'link_index': 0, 'source': 8, 'target': 21, 'start_time': 3, 'end_time': 4, 'queue_delay': 0, 'transmission_delay': 1, 'processing_delay': 0, 'propagation_delay': 5, 'min_bandwidth': 56.0, 'max_bandwidth': 56.0, 'avg_bandwidth': 56.0, 'data_input': 50, 'data_output': 50}, {'hop_index': 2, 'link_index': 1, 'source': 21, 'target': 16, 'start_time': 4, 'end_time': 5, 'queue_delay': 0, 'transmission_delay': 1, 'processing_delay': 0, 'propagation_delay': 3, 'min_bandwidth': 70.0, 'max_bandwidth': 70.0, 'avg_bandwidth': 70.0, 'data_input': 50, 'data_output': 20}, {'hop_index': 3, 'link_index': 0, 'source': 16, 'target': 21, 'start_time': 5, 'end_time': 6, 'queue_delay': 0, 'transmission_delay': 1, 'processing_delay': 0, 'propagation_delay': 3, 'min_bandwidth': 70.0, 'max_bandwidth': 70.0, 'avg_bandwidth': 70.0, 'data_input': 20, 'data_output': 20}, {'hop_index': 3, 'link_index': 1, 'source': 21, 'target': 24, 'start_time': 6, 'end_time': 7, 'queue_delay': 0, 'transmission_delay': 1, 'processing_delay': 0, 'propagation_delay': 3, 'min_bandwidth': 63.0, 'max_bandwidth': 63.0, 'avg_bandwidth': 63.0, 'data_input': 20, 'data_output': 20}, {'hop_index': 3, 'link_index': 2, 'source': 24, 'target': 2, 'start_time': 7, 'end_time': 8, 'queue_delay': 0, 'transmission_delay': 1, 'processing_delay': 0, 'propagation_delay': 3, 'min_bandwidth': 37.0, 'max_bandwidth': 37.0, 'avg_bandwidth': 37.0, 'data_input': 20, 'data_output': 1}, {'hop_index': 4, 'link_index': 0, 'source': 2, 'target': 10, 'start_time': 8, 'end_time': 9, 'queue_delay': 0, 'transmission_delay': 1, 'processing_delay': 0, 'propagation_delay': 3, 'min_bandwidth': 37.0, 'max_bandwidth': 37.0, 'avg_bandwidth': 37.0, 'data_input': 1, 'data_output': 1}, {'hop_index': 4, 'link_index': 1, 'source': 10, 'target': 4, 'start_time': 9, 'end_time': 10, 'queue_delay': 0, 'transmission_delay': 1, 'processing_delay': 0, 'propagation_delay': 3, 'min_bandwidth': 57.0, 'max_bandwidth': 57.0, 'avg_bandwidth': 57.0, 'data_input': 1, 'data_output': 1}, {'hop_index': 4, 'link_index': 2, 'source': 4, 'target': 8, 'start_time': 10, 'end_time': 11, 'queue_delay': 0, 'transmission_delay': 1, 'processing_delay': 0, 'propagation_delay': 4, 'min_bandwidth': 45.0, 'max_bandwidth': 45.0, 'avg_bandwidth': 45.0, 'data_input': 1, 'data_output': 0}]"


In [13]:
# Estraiamo gli hops del primo pacchetto finito
hops_list = df_finished.iloc[0]["Hops"]
df_hops = pd.DataFrame(hops_list)

# Selezioniamo le colonne chiave per capire il percorso e i ritardi
cols = ["hop_index", "link_index", "source", "target", "propagation_delay", "transmission_delay", "data_input", "data_output"]
df_hops[cols]

,hop_index,link_index,source,target,propagation_delay,transmission_delay,data_input,data_output
0,0,0,8,8,0,0,75,75
1,1,0,8,8,0,0,75,50
2,2,0,8,21,5,1,50,50
3,2,1,21,16,3,1,50,20
4,3,0,16,21,3,1,20,20
5,3,1,21,24,3,1,20,20
6,3,2,24,2,3,1,20,1
7,4,0,2,10,3,1,1,1
8,4,1,10,4,3,1,1,1
9,4,2,4,8,4,1,1,0
